# Эксперименты с моделями зарплаты Avito

Выполните все ячейки. Ноутбук восстановит зафиксированный снимок данных, обучит базу и вариант с одним изменением, отдельно проверит готовый `global.joblib` и запишет один `experiment.json` в `ML/experiments/<EXPERIMENT_ID>/`.

Из корня репозитория установите зависимости командой `python -m pip install -r ML/models/2026-11-18_priority_salary_gold/requirements.txt` и используйте это окружение как ядро Jupyter. Исходное сравнение схем и его артефакты доступны в [архиве исследования](experiments/2026-11-18_salary_routing/README.md).

In [ ]:
# None создаёт новый ID при каждом выполнении. Можно задать своё имя прогона.
EXPERIMENT_ID = None

# Одно изменение. None запускает базу и проверку архивных весов.
CHANGE = {'text_features': True}
# Другой пример: CHANGE = {'training_target_rule': 'native_exact_only_v1'}

DATASET_CSV = None  # При необходимости укажите путь к точной копии снимка.
SAVE_WEIGHTS = True
WEIGHTS_FROM = None  # Путь к своему experiment.json для проверки ранее сохранённых пакетов.
TABLE_ROW_URL = None  # Ссылка на строку общей таблицы с этим EXPERIMENT_ID.


## База и проверка

Общий CatBoost из PR № 180 предсказывает P25, P50 и P75 в исходной единице оплаты. База и гипотеза используют одни и те же три отложенных фолда. Работодатели и группы дублей изолированы между обучением, валидацией и тестом. Gold — приоритетные вакансии внутри соответствующего теста. Архивные веса `global.joblib` относятся к фолду 0.

Пример добавляет текстовые признаки. TF-IDF и SVD обучаются внутри каждого обучающего фолда на очищенных заголовках, описаниях и поле «Что получают работники». Зарплатные фрагменты и числа удаляются до кодирования. Для проверки правил нормализации можно передать `training_target_rule` в `CHANGE`. Реализация версионированных правил находится в [`training_rows`](salary_routing_experiment.py). При изменении обучающей цели тестовые целевые значения сохраняются.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
import sys
from uuid import uuid4

cwd = Path.cwd().resolve()
root = next(path for path in (cwd, *cwd.parents) if (path / 'ML/salary_routing_experiment.py').is_file())
if str(root / 'ML') not in sys.path:
    sys.path.insert(0, str(root / 'ML'))

from salary_routing_experiment import run_experiment

baseline = {
    'snapshot_revision': '2d9a65561f61f892dc6e5b05ece290d71bb95671',
    'dataset_sha256': 'a56e81b2b7f524da57ca93ce4c6fe984dae950d67ec98b9f37429e5fde76983e',
    'scheme': 'global',
    'global_iterations': 240,
    'depth': 6,
    'training_target_rule': 'native_exact_or_range_v1',
    'text_features': False,
}

run_id = EXPERIMENT_ID or f'{datetime.now(timezone.utc):%Y-%m-%d}_salary_routing_{uuid4().hex[:8]}'
journal_path = run_experiment(
    run_id, baseline, change=CHANGE,
    dataset=root / DATASET_CSV if DATASET_CSV else None,
    replay_path=root / WEIGHTS_FROM if WEIGHTS_FROM else None,
    save_weights=SAVE_WEIGHTS,
    table_row_url=TABLE_ROW_URL,
)
report = json.loads(journal_path.read_text(encoding='utf-8'))
print(journal_path.relative_to(root))


## Качество и охват

Охват теста — доля вакансий с прогнозом. Покрытие P25–P75 — доля наблюдаемых сумм внутри интервала среди получивших прогноз, с ориентиром 50%. Парные дельты считаются на вакансиях с прогнозом от обеих моделей. Сравнивайте их вместе с изменением охвата. Ошибки в рублях по единицам оплаты и налоговой базе есть в JSON.

In [ ]:
import pandas as pd
from IPython.display import display

table = []
for case, run in report['runs'].items():
    for scope, result in run['metrics'].items():
        quality = result['quality']
        table.append({
            'Запуск': case, 'Тест': scope, 'Прогнозов': quality['n'],
            'Размер теста': quality['available_n'], 'Норм. pinball': quality.get('scaled_pinball'),
            'Норм. MAE P50': quality.get('scaled_p50_mae'),
            'Покрытие P25–P75': quality.get('coverage'),
        })
display(pd.DataFrame(table).round(5))

times = [{'Запуск': case, **run['training_time']} for case, run in report['runs'].items()
         if run['training_time'] is not None]
if times:
    display(pd.DataFrame(times).rename(columns={'wall_seconds': 'Время по часам, с',
                                                'process_cpu_seconds': 'Время CPU процесса, с'}).round(2))

if report['comparison'] is not None:
    print(report['change'])
    display(pd.DataFrame({
        scope: {'Общих прогнозов': row['n'], **row['candidate_minus_baseline'],
                '95% ДИ для Δ pinball': row['scaled_pinball_delta_group_bootstrap_95pct'],
                'Δ охвата теста': row['test_prediction_share_delta']}
        for scope, row in report['comparison'].items()
    }).T)


## Журнал и повторный запуск

В Git сохраняйте `experiment.json`. В нём есть ID обучения, валидации и теста, SHA-256 и ревизия снимка, версии правил и кода, признаки, модели, параметры, маршруты, метрики и ссылки с контрольными суммами на веса. Списки ID находятся в JSON. Данные восстанавливаются из зафиксированной ревизии репозитория.
Общая [таблица запусков](experiments/README.md) собирается из паспортов командой `python ML/update_experiment_index.py` из корня репозитория. Командные пакеты находятся в `ML/models/<EXPERIMENT_ID>/`.

Для недельного прогона задайте `EXPERIMENT_ID`, совпадающий с ID записи в общей таблице, и внесите ссылку на эту запись в `TABLE_ROW_URL`. В паспорт также попадут ссылка на архивный пакет базы, параметры обучения и seed, CPU и ОС, версии Python и библиотек, время обучения по каждому фолду. Время по часам показывает длительность обучения. Время CPU процесса суммирует работу его потоков и при параллельном обучении может быть больше. Для проверки готовых весов время обучения в новом отчёте пустое.

Новые веса вместе с обработкой признаков сохраняются в `ML/.cache/salary_routing_weights/<EXPERIMENT_ID>/`. Для проверки своего прогона задайте путь к его JSON в `WEIGHTS_FROM` и выполните все ячейки. Ноутбук сверит ID, SHA-256 весов и метрики и запишет новый отчёт. Используйте пакеты, которым доверяете. Архивный `global.joblib` доступен в Git. Для повторной проверки нового пакета на другой машине скопируйте каталог весов. Модель также можно переобучить по снимку и записанным параметрам.
Рядом с новыми пакетами сохраняется копия их JSON-паспорта.

Для новой гипотезы оставьте `WEIGHTS_FROM = None` и поменяйте одно значение в `CHANGE`. Доступны `text_features`, `training_target_rule`, `scheme`, `depth` и `global_iterations`. `scheme` принимает `global`, `by_mode`, `by_segment`, `hybrid`, `empirical` или `manual`.